In [ ]:
!pip install -q ezdxf

In [ ]:
import ezdxf

from pathlib import Path

import json

import subprocess

import os

In [ ]:
# ======================================================
# Load State
# ======================================================


ROOT = Path.cwd().parent


STATE_FILE = (

ROOT /

"config" /

"project_state.json"

)


with open(

    STATE_FILE,

    encoding="utf-8"

) as f:


    PROJECT_STATE=json.load(f)



PROJECT_STATE

In [ ]:
# ======================================================
# DXF Input
# ======================================================


DXF_FILE = Path(

    PROJECT_STATE["dxf_file"]

)


if not DXF_FILE.exists():

    raise Exception(

        "DXF not found"

    )


print(

DXF_FILE

)

In [ ]:
# ======================================================
# DXF Check
# ======================================================


try:


    doc = ezdxf.readfile(

        DXF_FILE

    )


    print(

    "DXF Valid"

    )


except Exception as e:


    raise Exception(e)

In [ ]:
# ======================================================
# Entity Report
# ======================================================


msp = doc.modelspace()


entity_count={}



for entity in msp:


    name=entity.dxftype()


    entity_count[name]=(

        entity_count.get(name,0)

        +

        1

    )



entity_count

In [ ]:
# ======================================================
# Download ODA Converter
# ======================================================


!wget -q https://download.opendesign.com/guestfiles/ODAFileConverter_QT5_lnxX64_26.6dll.deb \
-O oda.deb


!dpkg -i oda.deb || true

In [ ]:
# ======================================================
# DXF TO DWG
# ======================================================


def convert_to_dwg(dxf_file):


    output_dir = (

        dxf_file.parent

    )


    command=f"""

    ODAFileConverter \

    "{dxf_file.parent}" \

    "{output_dir}" \

    "ACAD2018" \

    "DWG" \

    0 \

    1

    """


    result=subprocess.run(

        command,

        shell=True,

        capture_output=True,

        text=True

    )


    print(

    result.stdout

    )


    return result

In [ ]:
result = convert_to_dwg(

    DXF_FILE

)


print(

result.returncode

)

In [ ]:
# ======================================================
# Locate DWG
# ======================================================


dwg_files=list(

    DXF_FILE.parent.glob(

        "*.dwg"

    )

)



if len(dwg_files):


    DWG_FILE=dwg_files[0]


    print(

    "DWG:",

    DWG_FILE

    )


else:


    print(

    "DWG not created"

    )

In [ ]:
# ======================================================
# Update Pipeline
# ======================================================


if "DWG_FILE" in globals():


    PROJECT_STATE.update({

        "dwg_file":

        str(DWG_FILE)

    })



with open(

    STATE_FILE,

    "w",

    encoding="utf-8"

) as f:


    json.dump(

        PROJECT_STATE,

        f,

        indent=4

    )


print(

"DWG State Updated"

)